# Curation Errors in MoleculeNet

In his [blog
post](https://practicalcheminformatics.blogspot.com/2023/08/we-need-better-benchmarks-for-machine.html),
Pat Walters points out that there are serious curation errors in the MoleculeNet
benchmark dataset that are often overlooked in literature. In this notebook, we
will attempt to verify his claims by downloading the binary classification blood
brain barrier penetration (BBBP) dataset and checking for the presence of the
mentioned errors. 


### Main Claims

The main claims regarding the curation errors in the MoleculeNet benchmark
dataset are threefold:

1. **Duplicate Entries**: There are 59 duplicate entries in the BBBP dataset.
2. **Inconsistent labels**: Among the duplicate entries, 10 are labeled as both penetrant and non-penetrant at the same time.
3. **Incorrect labels**: Some entries (e.g.,  glyburide) are labeled as
   penetrant while [literature](https://doi.org/10.1212/WNL.0000000000007378) suggests otherwise.

The duplicate entries can lead to data leakage and overfitting, while the
inconsistent and incorrect labels can mislead the training process and
evaluation of machine learning models. Therefore, it is crucial to identify and
address these issues to ensure the reliability of the benchmark dataset.

### Downloading the dataset

In [ ]:
# Install deepchem with `pip install deepchem`
# and download it using deepchem api
# Use `data_dir` argument can be used to specify the location of the csv file
from deepchem.molnet import load_bbbp
bbbp = load_bbbp(data_dir="./data")

In [ ]:
# The second way is to download the data directly
!wget -P ./data https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv

### Load the dataset

In [ ]:
import os
import csv
from collections import defaultdict, Counter

import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor

try:
    from rdkit.Chem.Draw import IPythonConsole
except Exception:
    pass

In [ ]:
# Set csv_path to that file.
csv_path = "./data/BBBP.csv"  # <-- change this to your actual path

# Create output directory if it doesn't exist
out_dir = "outputs"  # directory to save outputs
os.makedirs(out_dir, exist_ok=True)

# Load the dataset using pandas
df = pd.read_csv(csv_path)

smiles_col = "smiles"  # DeepChem's BBBP file uses 'smiles'
label_col = "p_np"     # binary BBBP label in MoleculeNet (1: penetrant, 0: non-penetrant)

# Basic sanity check
assert smiles_col in df.columns, f"SMILES column '{smiles_col}' not found in CSV"
assert label_col in df.columns, f"Label column '{label_col}' not found in CSV"

# Add original row index as a column for traceability
df = df.reset_index(drop=False).rename(columns={"index": "original_row"})

### Canonicalize the SMILES with RDKit

In [ ]:
# Function to canonicalize SMILES and handle invalid entries
def canonicalize_smiles(s):
    if pd.isna(s):
        return None
    mol = Chem.MolFromSmiles(str(s))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

# Canonicalize the SMILES and create a new column for it
df["canonical_smiles"] = df[smiles_col].apply(canonicalize_smiles)

# Separate valid and invalid entries based on the presence of canonical SMILES
invalid_rows = df[df["canonical_smiles"].isna()].copy()
valid_df = df.dropna(subset=["canonical_smiles"]).copy()

### Count the duplicates

In [ ]:
# Count the occurrences of each canonical SMILES
canon_counts = Counter(valid_df["canonical_smiles"])

# Filter to only those canonical SMILES that have duplicates
duplicate_canon = {s: c for s, c in canon_counts.items() if c > 1}

# Take out all rows corresponding to duplicated canonical SMILES
duplicate_rows = valid_df[valid_df["canonical_smiles"].isin(duplicate_canon)].copy()

# Create a summary DataFrame that aggregates information about the duplicates
duplicate_summary = (
    duplicate_rows.groupby("canonical_smiles")
    .agg(
        count=("canonical_smiles", "size"),
        labels_present=(label_col, lambda x: sorted(set(int(v) for v in x if pd.notna(v)))),
        original_rows=("original_row", lambda x: list(x)),
        example_smiles=(smiles_col, "first"),
    )
    .reset_index()
    .sort_values(["count", "canonical_smiles"], ascending=[False, True])
)

### Identify the conflicting entries

In [ ]:
# Create the mapping: canonical_smiles -> set of labels
labels_by_canon = defaultdict(set)
rows_by_canon = defaultdict(list)

# Iterate through valid rows and populate the mappings
for _, row in valid_df.iterrows():
    s = row["canonical_smiles"]
    y = row[label_col]
    if pd.notna(y):
        labels_by_canon[s].add(int(y))
    rows_by_canon[s].append(int(row["original_row"]))

# Identify canonical SMILES that have more than one unique label
conflict_smiles = {
    s: sorted(list(labels))
    for s, labels in labels_by_canon.items()
    if len(labels) > 1
}

# Take out all rows corresponding to conflicting canonical SMILES
conflict_rows = valid_df[valid_df["canonical_smiles"].isin(conflict_smiles)].copy()

# Create a summary DataFrame for conflicting SMILES
conflict_summary = (
    conflict_rows.groupby("canonical_smiles")
    .agg(
        count=("canonical_smiles", "size"),
        labels_present=(label_col, lambda x: sorted(set(int(v) for v in x if pd.notna(v)))),
        original_rows=("original_row", lambda x: list(x)),
        example_smiles=(smiles_col, "first"),
    )
    .reset_index()
    .sort_values(["count", "canonical_smiles"], ascending=[False, True])
)

### Store the results

In [ ]:
# Save all outputs to CSV files
invalid_rows.to_csv(os.path.join(out_dir, "bbbp_invalid_smiles.csv"), index=False)
duplicate_rows.to_csv(os.path.join(out_dir, "bbbp_duplicate_rows.csv"), index=False)
duplicate_summary.to_csv(os.path.join(out_dir, "bbbp_duplicate_summary.csv"), index=False)
conflict_rows.to_csv(os.path.join(out_dir, "bbbp_conflict_rows.csv"), index=False)
conflict_summary.to_csv(os.path.join(out_dir, "bbbp_conflict_summary.csv"), index=False)

# Save the canonical SMILES with their counts (only those with duplicates) to a CSV file
with open(os.path.join(out_dir, "bbbp_duplicate_counts.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["canonical_smiles", "count"])
    for s, c in sorted(duplicate_canon.items(), key=lambda x: (-x[1], x[0])):
        writer.writerow([s, c])

### Define helper functions for visualization

In [ ]:
# Create molecule objects from SMILES for visualization
def mol_from_smiles_for_drawing(s):
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        return None
    rdDepictor.Compute2DCoords(mol)
    return mol

# Helper function to chunk a list into smaller pieces
def chunk_list(items, chunk_size):
    for i in range(0, len(items), chunk_size):
        yield items[i:i + chunk_size]

# Helper function to write RDKit image objects to file
def write_rdkit_image(obj, out_path):
    # Case 1: raw PNG bytes
    if isinstance(obj, (bytes, bytearray)):
        with open(out_path, "wb") as f:
            f.write(obj)
        return

    # Case 2: Jupyter/IPython display object with .data
    if hasattr(obj, "data"):
        data = obj.data
        if isinstance(data, str):
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(data)
            return
        if isinstance(data, (bytes, bytearray)):
            with open(out_path, "wb") as f:
                f.write(data)
            return

    # Case 3: PIL image-like object
    if hasattr(obj, "save"):
        obj.save(out_path)
        return

    raise TypeError(f"Unsupported image object type: {type(obj)}")

# Helper function to save a grid of molecule images to PNG
def save_grid_png(mols, legends, out_path, mols_per_row=5, sub_img_size=(300, 300)):
    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=mols_per_row,
        subImgSize=(int(sub_img_size[0]), int(sub_img_size[1])),
        legends=legends,
        useSVG=False,
        returnPNG=True
    )
    write_rdkit_image(img, out_path)

### Draw the duplicate structures

In [ ]:
# Sort the duplicate canonical SMILES by count (descending) and then alphabetically
dup_smiles_sorted = sorted(duplicate_canon.items(), key=lambda x: (-x[1], x[0]))
dup_mols = []
dup_legends = []

# Create molecule objects and legends for each duplicate SMILES
for smi, count in dup_smiles_sorted:
    mol = mol_from_smiles_for_drawing(smi)
    if mol is not None:
        labels = sorted(list(labels_by_canon.get(smi, [])))
        dup_mols.append(mol)
        dup_legends.append(f"count={count} | labels={labels}\n{smi}")

# Save the duplicate structures in a grid format
for page_idx, mol_chunk in enumerate(chunk_list(dup_mols, 25), start=1):
    start = (page_idx - 1) * 25
    end = page_idx * 25
    legend_chunk = dup_legends[start:end]

    save_grid_png(
        mol_chunk,
        legend_chunk,
        os.path.join(out_dir, f"bbbp_duplicate_structures_page_{page_idx}.png"),
        mols_per_row=5,
        sub_img_size=(300, 300)
    )

### Draw the conflicting molecules

In [ ]:
# Sort the conflicting canonical SMILES alphabetically
conflict_smiles_sorted = sorted(conflict_smiles.items(), key=lambda x: x[0])
conflict_mols = []
conflict_legends = []

# Create molecule objects and legends for each conflicting SMILES
for smi, labels in conflict_smiles_sorted:
    mol = mol_from_smiles_for_drawing(smi)
    if mol is not None:
        count = duplicate_canon.get(smi, 1)
        conflict_mols.append(mol)
        conflict_legends.append(f"labels={labels} | count={count}\n{smi}")

# Save the conflicting structures in a grid format
for page_idx, mol_chunk in enumerate(chunk_list(conflict_mols, 25), start=1):
    start = (page_idx - 1) * 25
    end = page_idx * 25
    legend_chunk = conflict_legends[start:end]

    save_grid_png(
        mol_chunk,
        legend_chunk,
        os.path.join(out_dir, f"bbbp_conflict_structures_page_{page_idx}.png"),
        mols_per_row=5,
        sub_img_size=(300, 300)
    )

### Create a folder with individual images of conflicting molecules

In [ ]:
# Create a folder with individual images of conflicting molecules
single_dir = os.path.join(out_dir, "conflict_singletons")
os.makedirs(single_dir, exist_ok=True)

# Save each conflicting molecule as an individual PNG file with a filename that includes the SMILES and labels
for smi, labels in conflict_smiles.items():
    mol = mol_from_smiles_for_drawing(smi)
    if mol is None:
        continue

    safe_name = "".join(c if c.isalnum() else "_" for c in smi)[:120]
    out_file = os.path.join(single_dir, f"{safe_name}_labels_{'_'.join(map(str, labels))}.png")
    legend = f"labels={labels}\n{smi}"
    img = Draw.MolToImage(mol, size=(500, 400), legend=legend)
    write_rdkit_image(img, out_file)

### Summary

In [ ]:
print(f"Total rows: {len(df)}")
print(f"Valid canonical SMILES: {len(valid_df)}")
print(f"Invalid SMILES: {len(invalid_rows)}")
print(f"Distinct duplicated canonical SMILES: {len(duplicate_canon)}")
print(f"Distinct conflicting-label canonical SMILES: {len(conflict_smiles)}")
print(f"Output directory: {os.path.abspath(out_dir)}")